# ⚙️ Step-by-Step Feature Engineering Notebook

In this notebook, we transform and enrich the cleaned credit loan dataset by engineering informative financial features **one by one** using simple, clear code.

### Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

### Step 2: Load the Cleaned Dataset

In [ ]:
# Load cleaned data (handles both notebook folder and root execution)
data_path = '../data/processed/cleaned_data.csv' if os.path.exists('../data/processed/cleaned_data.csv') else 'data/processed/cleaned_data.csv'
df = pd.read_csv(data_path)

print(f'Dataset Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

### Step 3: Loan-to-Income (LTI) Ratio
- Measures the total borrowed loan amount relative to the borrower's annual income.

In [ ]:
df['loan_to_income'] = (df['loan_amount'] / df['income']).round(4)

print('Loan-to-Income Summary:')
print(df['loan_to_income'].describe().round(4))
df[['loan_amount', 'income', 'loan_to_income']].head()

### Step 4: Remaining Annual Disposable Income
- Calculates the remaining income left after deducting the loan amount.

In [ ]:
df['remaining_income'] = df['income'] - df['loan_amount']

print('Remaining Income Summary:')
print(df['remaining_income'].describe().round(2))
df[['income', 'loan_amount', 'remaining_income']].head()

### Step 5: Estimated Annual Interest Payment
- Calculates the yearly interest cost: `loan_amount * (interest_rate / 100)`.

In [ ]:
df['annual_interest_payment'] = (df['loan_amount'] * (df['interest_rate'] / 100)).round(2)

print('Annual Interest Payment Summary:')
print(df['annual_interest_payment'].describe().round(2))
df[['loan_amount', 'interest_rate', 'annual_interest_payment']].head()

### Step 6: Estimated Monthly Debt Payment
- Estimates approximate monthly payment assuming a standard 36-month loan term.

In [ ]:
df['estimated_monthly_payment'] = ((df['loan_amount'] + (df['annual_interest_payment'] * 3)) / 36).round(2)

print('Estimated Monthly Payment Summary:')
print(df['estimated_monthly_payment'].describe().round(2))
df[['loan_amount', 'interest_rate', 'estimated_monthly_payment']].head()

### Step 7: Monthly Debt Payment to Monthly Income Ratio (DTI)
- Shows the percentage of monthly salary committed toward servicing the loan.

In [ ]:
df['monthly_debt_to_income'] = (df['estimated_monthly_payment'] / (df['income'] / 12)).round(4)

print('Monthly Debt-to-Income Summary:')
print(df['monthly_debt_to_income'].describe().round(4))
df[['estimated_monthly_payment', 'income', 'monthly_debt_to_income']].head()

### Step 8: Credit History to Age Ratio
- Measures the fraction of the borrower's life spent establishing credit history.

In [ ]:
df['credit_to_age_ratio'] = (df['credit_history_years'] / df['age']).round(4)

print('Credit History to Age Ratio Summary:')
print(df['credit_to_age_ratio'].describe().round(4))
df[['credit_history_years', 'age', 'credit_to_age_ratio']].head()

### Step 9: Employment Length to Age Ratio
- Reflects employment longevity and stability relative to the person's age.

In [ ]:
df['emp_to_age_ratio'] = (df['employment_years'] / df['age']).round(4)

print('Employment to Age Ratio Summary:')
print(df['emp_to_age_ratio'].describe().round(4))
df[['employment_years', 'age', 'emp_to_age_ratio']].head()

### Step 10: Loan Amount per Employment Year
- Evaluates loan scale compared to years in the workforce.

In [ ]:
df['loan_per_emp_year'] = (df['loan_amount'] / (df['employment_years'] + 1)).round(2)

print('Loan per Employment Year Summary:')
print(df['loan_per_emp_year'].describe().round(2))
df[['loan_amount', 'employment_years', 'loan_per_emp_year']].head()

### Step 11: Binary Flag - High Debt Burden (> 30% of Income)
- Flags high-risk borrowers where the loan exceeds 30% of their annual salary.

In [ ]:
df['high_debt_flag'] = (df['loan_income_ratio'] > 0.30).astype(int)

print('High Debt Burden Count (0 = Normal, 1 = High Debt):')
print(df['high_debt_flag'].value_counts())

### Step 12: Binary Flag - High Interest Rate (> 15%)
- Identifies loans assigned high interest rates, typically given to higher risk profiles.

In [ ]:
df['high_interest_flag'] = (df['interest_rate'] > 15.0).astype(int)

print('High Interest Flag Count (0 = <=15%, 1 = >15%):')
print(df['high_interest_flag'].value_counts())

### Step 13: Binary Flag - Previous Default Indicator
- Converts 'Y' and 'N' text values into clean machine learning numeric flags (0 or 1).

In [ ]:
df['has_previous_default'] = (df['previous_default'] == 'Y').astype(int)

print('Previous Default Binary Count (0 = Clean Record, 1 = Past Default):')
print(df['has_previous_default'].value_counts())

### Step 14: Binary Flag - Young Borrower Flag (Age < 25)
- Identifies early-career applicants with potentially less financial cushion.

In [ ]:
df['is_young_borrower'] = (df['age'] < 25).astype(int)

print('Young Borrower Count (0 = 25+, 1 = Under 25):')
print(df['is_young_borrower'].value_counts())

### Step 15: Log Transformations for Skewed Financial Features
- Log-transform `income` and `loan_amount` to normalize skewed distributions for linear models.

In [ ]:
df['log_income'] = np.log1p(df['income']).round(4)
df['log_loan_amount'] = np.log1p(df['loan_amount']).round(4)

print('Log Transformation Summary:')
print(df[['income', 'log_income', 'loan_amount', 'log_loan_amount']].head())

### Step 16: Age Group Cohort Bins
- Categorizes borrowers into meaningful life stages: Young (18-25), Adult (26-40), Middle-Age (41-60), Senior (>60).

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 25, 40, 60, 100],
    labels=['Young', 'Adult', 'Middle-Age', 'Senior']
)

print('Age Group Distribution:')
print(df['age_group'].value_counts())

### Step 17: Income Quartile Bins
- Segments borrowers into 4 equal quartiles: Low, Medium, Upper, and High Income.

In [ ]:
df['income_quartile'] = pd.qcut(
    df['income'],
    q=4,
    labels=['Low Income', 'Medium Income', 'Upper Income', 'High Income']
)

print('Income Quartile Distribution:')
print(df['income_quartile'].value_counts())

### Step 18: Loan Size Categories
- Groups requested loan amounts into Small, Medium, Large, and Jumbo tiers.

In [ ]:
df['loan_size_category'] = pd.cut(
    df['loan_amount'],
    bins=[0, 5000, 15000, 25000, 100000],
    labels=['Small', 'Medium', 'Large', 'Jumbo']
)

print('Loan Size Category Distribution:')
print(df['loan_size_category'].value_counts())

### Step 19: Ordinal Encoding - Loan Grade (A=1 to G=7)
- Maps credit rating grades A through G to risk scores where 1 is prime and 7 is highest risk.

In [ ]:
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['loan_grade_num'] = df['loan_grade'].map(grade_map)

print('Loan Grade Ordinal Mapping Sample:')
print(df[['loan_grade', 'loan_grade_num']].drop_duplicates().sort_values('loan_grade_num'))

### Step 20: One-Hot Encoding for Categorical Features
- Creates dummy indicator columns for nominal features `home_ownership` and `loan_purpose`.

In [ ]:
dummies = pd.get_dummies(df[['home_ownership', 'loan_purpose']], drop_first=True, dtype=int)

print('One-Hot Encoded Features Created:')
print(list(dummies.columns))
dummies.head()

### Step 21: Correlation of Engineered Features with Loan Default
- Evaluates the linear relationship between each new numeric feature and the target (`default`).

In [ ]:
numeric_cols = [
    'loan_to_income',
    'loan_income_ratio',
    'remaining_income',
    'annual_interest_payment',
    'estimated_monthly_payment',
    'monthly_debt_to_income',
    'credit_to_age_ratio',
    'emp_to_age_ratio',
    'loan_per_emp_year',
    'high_debt_flag',
    'high_interest_flag',
    'has_previous_default',
    'is_young_borrower',
    'loan_grade_num',
    'default'
]

corr_series = df[numeric_cols].corr()['default'].sort_values(ascending=False)
print('Correlation of Features with Default:')
print(corr_series.round(4))

### Step 22: Save the Fully Featured Dataset
- Saves the complete dataset with all new engineered features to `data/processed/featured_data.csv`.

In [ ]:
featured_path = '../data/processed/featured_data.csv' if os.path.exists('../data') else 'data/processed/featured_data.csv'
df.to_csv(featured_path, index=False)

print(f'Full Featured Data Saved to: {featured_path}')
print(f'Final Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')

### Step 23: Save Baseline Model Dataset for Modeling Pipelines
- Exports clean predictors and target to `data/processed/model_data.csv` for downstream model training.

In [ ]:
model_cols = [
    'age',
    'income',
    'employment_years',
    'home_ownership',
    'loan_amount',
    'loan_purpose',
    'credit_history_years',
    'default'
]

model_df = df[model_cols].copy()
model_path = '../data/processed/model_data.csv' if os.path.exists('../data') else 'data/processed/model_data.csv'
model_df.to_csv(model_path, index=False)

print('=' * 60)
print(f'Model Dataset Successfully Saved to: {model_path}')
print(f'Dimensions: {model_df.shape[0]} rows, {model_df.shape[1]} columns')
print('=' * 60)